# Text Clustering Using K-Means and Non-negative Matrix Factorization (NMF)

This project investigates text clustering using two methods: K-Means and Non-negative Matrix Factorization (NMF). Four datasets are constructed from the TDT2 and Reuters21578 collections according to class frequencies. The clustering results are evaluated using Purity and normalized Entropy.

## 1. Required Libraries

Install and import the libraries required for loading MATLAB files, numerical computation, data processing, and clustering.

In [ ]:
!pip install pymatreader

import numpy as np
import pandas as pd
from pymatreader import read_mat
from scipy.sparse import issparse, vstack
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF

## 2. Dataset Construction

The MATLAB datasets contain two main variables:

- `gnd`: The ground-truth class labels.
- `fea`: The feature matrix representing the documents.

The datasets are constructed by first calculating the frequency of each class and sorting the classes in descending order of frequency. Documents are then selected according to the specified sampling percentages.

For each selected class, documents are taken in their original dataset order. The selected feature vectors and their corresponding labels are stored separately.

In [ ]:
def get_class_information(data):
    labels = np.asarray(data['gnd']).ravel()
    classes, counts = np.unique(labels, return_counts=True)

    # Sort classes by descending frequency
    sorted_indices = np.argsort(-counts, kind='stable')
    sorted_classes = classes[sorted_indices]
    sorted_counts = counts[sorted_indices]

    return labels, sorted_classes, sorted_counts


def construct_dataset(data, selected_classes, percentages):
    labels = np.asarray(data['gnd']).ravel()
    features = data['fea']

    selected_features = []
    selected_labels = []

    for class_label, percentage in zip(selected_classes, percentages):
        class_indices = np.where(labels == class_label)[0]
        number_to_select = int(len(class_indices) * percentage)
        selected_indices = class_indices[:number_to_select]

        if len(selected_indices) > 0:
            selected_features.append(features[selected_indices])
            selected_labels.append(labels[selected_indices])

    if not selected_features:
        raise ValueError('No documents were selected.')

    if issparse(features):
        X = vstack(selected_features).toarray()
    else:
        X = np.vstack(selected_features)

    y = np.concatenate(selected_labels)

    return X, y


def display_dataset_info(name, X, y):
    print(name)
    print('Feature matrix shape:', X.shape)
    print('Number of documents:', len(y))
    print('Number of classes:', len(np.unique(y)))
    print('Class distribution:')
    print(pd.Series(y).value_counts().sort_index())
    print('------------------------------')

## 3. TDT2 Dataset Preparation

The TDT2 dataset is loaded from the MATLAB file. Its classes are sorted according to their frequencies.

Two datasets are constructed:

- **Tdt2_top10:** Select 10% of the documents from each of the ten most frequent classes.
- **Tdt2_last10:** Select all documents from each of the ten least frequent classes.

The selected documents and their original class labels are stored in separate variables.

In [ ]:
# Load TDT2 Dataset
tdt2_data = read_mat('/content/TDT2 (1).mat')

# Calculate Class Frequencies
tdt2_labels, tdt2_sorted_classes, tdt2_sorted_counts = get_class_information(tdt2_data)

print('TDT2 class frequencies (descending):')
print(list(zip(tdt2_sorted_classes, tdt2_sorted_counts)))

# Tdt2_top10: 10% of the ten most frequent classes
tdt2_top10_classes = tdt2_sorted_classes[:10]
tdt2_top10_percentages = [0.10] * 10

matrix_tdt2_top10, label_tdt2_top10 = construct_dataset(
    tdt2_data,
    tdt2_top10_classes,
    tdt2_top10_percentages
)

# Tdt2_last10: All documents from the ten least frequent classes
tdt2_last10_classes = tdt2_sorted_classes[-10:]
tdt2_last10_percentages = [1.0] * 10

matrix_tdt2_last10, label_tdt2_last10 = construct_dataset(
    tdt2_data,
    tdt2_last10_classes,
    tdt2_last10_percentages
)

# Display Dataset Information
display_dataset_info('Tdt2_top10', matrix_tdt2_top10, label_tdt2_top10)
display_dataset_info('Tdt2_last10', matrix_tdt2_last10, label_tdt2_last10)

## 4. Reuters21578 Dataset Preparation

The Reuters21578 dataset is loaded and its classes are sorted by descending frequency.

Two datasets are constructed:

- **Reu_top10:** From the ten most frequent classes, select 5% of the first class, 10% of the second class, and 40% of each class from the third to the tenth.
- **Reu_top20:** Select 5% of the first class, 10% of the second class, and all documents from the third to the twentieth class.

The corresponding feature matrices and ground-truth labels are stored for clustering.

In [ ]:
# Load Reuters21578 Dataset
reu_data = read_mat('/content/Reuters21578.mat')

# Calculate Class Frequencies
reu_labels, reu_sorted_classes, reu_sorted_counts = get_class_information(reu_data)

print('Reuters class frequencies (descending):')
print(list(zip(reu_sorted_classes, reu_sorted_counts)))

# Reu_top10: 5%, 10%, and 40% sampling strategy
reu_top10_classes = reu_sorted_classes[:10]
reu_top10_percentages = [0.05, 0.10] + [0.40] * 8

matrix_Reu_top10, label_Reu_top10 = construct_dataset(
    reu_data,
    reu_top10_classes,
    reu_top10_percentages
)

# Reu_top20: 5%, 10%, and 100% sampling strategy
reu_top20_classes = reu_sorted_classes[:20]
reu_top20_percentages = [0.05, 0.10] + [1.0] * 18

matrix_Reu_top20, label_Reu_top20 = construct_dataset(
    reu_data,
    reu_top20_classes,
    reu_top20_percentages
)

# Display Dataset Information
display_dataset_info('Reu_top10', matrix_Reu_top10, label_Reu_top10)
display_dataset_info('Reu_top20', matrix_Reu_top20, label_Reu_top20)

## 5. K-Means Clustering

K-Means is a partition-based clustering algorithm. It starts with an initial set of cluster centers and assigns each document to the cluster whose centroid is closest to it. The cluster centers are then updated iteratively until the assignments converge or the maximum number of iterations is reached.

The number of clusters is set equal to the number of ground-truth classes in each dataset. Therefore, the Reuters top-20 dataset uses 20 clusters, while the other datasets use 10 clusters.

In [ ]:
def kmeans_clustering(X, n_clusters):
    model = KMeans(
        n_clusters=n_clusters,
        random_state=0,
        n_init=10
    )
    return model.fit_predict(X)


# Run K-Means on all four datasets
y_pred_top10_tdt2_kmeans = kmeans_clustering(matrix_tdt2_top10, 10)
y_pred_last10_tdt2_kmeans = kmeans_clustering(matrix_tdt2_last10, 10)
y_pred_top10_Reu_kmeans = kmeans_clustering(matrix_Reu_top10, 10)
y_pred_top20_Reu_kmeans = kmeans_clustering(matrix_Reu_top20, 20)

print('K-Means clustering completed.')

## 6. Non-negative Matrix Factorization (NMF)

NMF approximates a non-negative matrix X as the product of two non-negative matrices:

$$X \approx WH$$

In this implementation, the input matrix is transposed so that each document is represented by a column. NMF produces the matrices W and H, where H contains the component activations for each document.

Each document is assigned to the component with the largest activation value in its corresponding column of H. The number of components is set equal to the number of ground-truth classes.

In [ ]:
def nmf_clustering(X, n_components):
    if np.any(X < 0):
        raise ValueError('NMF requires non-negative input data.')

    model = NMF(
        n_components=n_components,
        init='random',
        random_state=0,
        max_iter=400
    )

    # Transpose so that documents are represented by columns
    X_transposed = X.T

    W = model.fit_transform(X_transposed)
    H = model.components_

    # Assign each document to the component with maximum activation
    predicted_labels = np.argmax(H, axis=0)

    return predicted_labels


# Run NMF on all four datasets
y_pred_top10_tdt2_nmf = nmf_clustering(matrix_tdt2_top10, 10)
y_pred_last10_tdt2_nmf = nmf_clustering(matrix_tdt2_last10, 10)
y_pred_top10_Reu_nmf = nmf_clustering(matrix_Reu_top10, 10)
y_pred_top20_Reu_nmf = nmf_clustering(matrix_Reu_top20, 20)

print('NMF clustering completed.')

## 7. Clustering Evaluation

The clustering results are evaluated using Purity and normalized Entropy. Since the cluster identifiers produced by clustering algorithms are arbitrary, the evaluation compares the distribution of true classes within each predicted cluster.

**Purity:** Measures the proportion of documents assigned to the most frequent true class within each cluster. Higher Purity indicates better clustering quality.

$$Purity = \frac{1}{N}\sum_i \max_j(n_{ij})$$

**Normalized Entropy:** Measures the uncertainty in the class distribution within clusters. Lower Entropy indicates more homogeneous clusters and generally better clustering quality.

$$Entropy = -\frac{1}{N\log(C)}\sum_i\sum_j n_{ij}\log\left(\frac{n_{ij}}{n_i}\right)$$

Here, $n_{ij}$ is the number of documents belonging to true class j in predicted cluster i, $n_i$ is the size of cluster i, N is the total number of documents, and C is the number of true classes.

In [ ]:
def evaluate_clustering(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    if len(y_true) != len(y_pred):
        raise ValueError('The true and predicted labels must have equal lengths.')

    true_classes = np.unique(y_true)
    predicted_clusters = np.unique(y_pred)

    n = len(y_true)
    n_classes = len(true_classes)

    # Construct the contingency matrix
    contingency = np.zeros(
        (len(predicted_clusters), n_classes),
        dtype=np.int64
    )

    for i, cluster in enumerate(predicted_clusters):
        for j, class_label in enumerate(true_classes):
            contingency[i, j] = np.sum(
                (y_pred == cluster) & (y_true == class_label)
            )

    # Calculate Purity
    purity_score = np.sum(np.max(contingency, axis=1)) / n

    # Calculate normalized Entropy
    entropy_sum = 0.0
    for i in range(len(predicted_clusters)):
        cluster_size = np.sum(contingency[i])

        if cluster_size == 0:
            continue

        probabilities = contingency[i] / cluster_size
        nonzero_probabilities = probabilities[probabilities > 0]

        cluster_entropy = -np.sum(
            nonzero_probabilities * np.log(nonzero_probabilities)
        )

        entropy_sum += cluster_size * cluster_entropy

    entropy_score = entropy_sum / (n * np.log(n_classes))

    return purity_score, entropy_score

## 8. Evaluation of K-Means Results

Purity and Entropy are calculated for the four datasets clustered using K-Means.

In [ ]:
kmeans_results = {
    'Tdt2_top10': (label_tdt2_top10, y_pred_top10_tdt2_kmeans),
    'Tdt2_last10': (label_tdt2_last10, y_pred_last10_tdt2_kmeans),
    'Reu_top10': (label_Reu_top10, y_pred_top10_Reu_kmeans),
    'Reu_top20': (label_Reu_top20, y_pred_top20_Reu_kmeans)
}

kmeans_scores = {}

for dataset_name, (y_true, y_pred) in kmeans_results.items():
    purity_score, entropy_score = evaluate_clustering(y_true, y_pred)
    kmeans_scores[dataset_name] = {
        'Purity': purity_score,
        'Entropy': entropy_score
    }

    print(dataset_name)
    print('Purity:', round(purity_score, 4))
    print('Entropy:', round(entropy_score, 4))
    print('------------------------------')

## 9. Evaluation of NMF Results

Purity and Entropy are calculated for the four datasets clustered using NMF.

In [ ]:
nmf_results = {
    'Tdt2_top10': (label_tdt2_top10, y_pred_top10_tdt2_nmf),
    'Tdt2_last10': (label_tdt2_last10, y_pred_last10_tdt2_nmf),
    'Reu_top10': (label_Reu_top10, y_pred_top10_Reu_nmf),
    'Reu_top20': (label_Reu_top20, y_pred_top20_Reu_nmf)
}

nmf_scores = {}

for dataset_name, (y_true, y_pred) in nmf_results.items():
    purity_score, entropy_score = evaluate_clustering(y_true, y_pred)
    nmf_scores[dataset_name] = {
        'Purity': purity_score,
        'Entropy': entropy_score
    }

    print(dataset_name)
    print('Purity:', round(purity_score, 4))
    print('Entropy:', round(entropy_score, 4))
    print('------------------------------')

## 10. Comparison of Clustering Algorithms

The evaluation scores of K-Means and NMF are combined into a single table to facilitate comparison across the four datasets. Higher Purity and lower Entropy indicate better clustering results.

In [ ]:
comparison_rows = []

for dataset_name in kmeans_scores:
    comparison_rows.append({
        'Dataset': dataset_name,
        'K-Means Purity': kmeans_scores[dataset_name]['Purity'],
        'K-Means Entropy': kmeans_scores[dataset_name]['Entropy'],
        'NMF Purity': nmf_scores[dataset_name]['Purity'],
        'NMF Entropy': nmf_scores[dataset_name]['Entropy']
    })

comparison_table = pd.DataFrame(comparison_rows)
print(comparison_table.round(4).to_string(index=False))

## 11. Reported Results and Discussion

The following values are reproduced from the results table provided in the project description. The column order is interpreted as K-Means Entropy, K-Means Purity, NMF Entropy, and NMF Purity.

| Dataset | K-Means Entropy | K-Means Purity | NMF Entropy | NMF Purity |
|---|---:|---:|---:|---:|
| Tdt2_top10 | 0.54 | 0.52 | 0.43 | 0.65 |
| Tdt2_last10 | 0.64 | 0.36 | 0.34 | 0.62 |
| Reu_top10 | 0.68 | 0.40 | 0.34 | 0.69 |
| Reu_top20 | 0.63 | 0.36 | 0.47 | 0.53 |

**Note:** Under this interpretation of the table, NMF has higher Purity and lower Entropy than K-Means in all four datasets. This conflicts with the final statement in the supplied description, which claims that K-Means performs better. The original table headings or result ordering should therefore be checked before making a final conclusion.

The scores generated by the implementation above should also be compared with the reported values. Differences may arise from dataset ordering, sampling, initialization, preprocessing, or implementation details.